# EDA — Synthetic Persona Conversations (`datasets_sol_100`)

`datasets_sol_100/` holds LLM-generated conversations for four behavioural attributes —
**gullibility**, **rationality**, **seriousness**, **certainty_seeking** — each realised at
three intensity levels, produced by `gpt-5.6-sol` in a single batch on 2026-09-06. Every
conversation is stored as a `.txt` transcript (`HUMAN:`/`ASSISTANT:` turns) plus a sidecar
`.json` with generation metadata (attribute, level, topic, seed, the full generation prompt,
and a `leak_stems_found` self-check for trait-word leakage).

This notebook covers: corpus coverage and naming consistency, structural integrity
(role alternation, duplicates, topic collisions), trait-word leakage, and a length-confound
check across levels.

## 1 · Setup

In [1]:
import glob
import hashlib
import json
import re
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 200)
plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["axes.facecolor"] = "white"
plt.rcParams["savefig.facecolor"] = "white"
plt.rcParams["axes.grid"] = False

ROOT = Path("..")
DATA = ROOT / "datasets_sol_100"
ATTRS = ["gullibility", "rationality", "seriousness", "certainty_seeking"]

PALETTE = {"low": "#2a78d6", "medium": "#8a8a8a", "neutral": "#8a8a8a", "high": "#eb6834"}

## 2 · Load

Walk every `attribute/*.json` file, pair it with its `.txt` transcript, and parse the
transcript into turns. This gives one row per conversation with both the generation
metadata and derived text statistics (turn count, role alternation, word counts).

In [2]:
TURN_RE = re.compile(r"(HUMAN|ASSISTANT): (.*?)(?=\n(?:HUMAN|ASSISTANT):|\Z)", re.S)


def parse_record(json_path: Path) -> dict:
    with open(json_path) as f:
        meta = json.load(f)
    txt_path = json_path.with_suffix(".txt")
    text = txt_path.read_text(encoding="utf-8") if txt_path.exists() else ""
    turns = TURN_RE.findall(text)
    roles = [r for r, _ in turns]
    alt_ok = bool(roles) and roles[0] == "HUMAN" and all(
        roles[i] != roles[i + 1] for i in range(len(roles) - 1)
    )
    human_words = sum(len(t.split()) for r, t in turns if r == "HUMAN")
    asst_words = sum(len(t.split()) for r, t in turns if r == "ASSISTANT")
    return {
        "attribute": meta["attribute"],
        "level": meta["level"],
        "topic": meta["topic"],
        "index": meta["index"],
        "model": meta["model"],
        "seed": meta["seed"],
        "call_index": meta["call_index"],
        "num_turns_meta": meta["num_turns"],
        "leak_stems_found": tuple(meta.get("leak_stems_found") or ()),
        "txt_exists": txt_path.exists(),
        "n_turns": len(turns),
        "alt_ok": alt_ok,
        "human_words": human_words,
        "asst_words": asst_words,
        "total_words": human_words + asst_words,
        "text": text,
        "file": str(json_path),
    }


records = [parse_record(Path(p)) for p in sorted(glob.glob(str(DATA / "*/*.json")))]
df = pd.DataFrame(records)
print(f"{len(df)} conversations loaded across {df['attribute'].nunique()} attributes")
df.head(3)[["attribute", "level", "topic", "index", "call_index", "n_turns", "total_words"]]

858 conversations loaded across 4 attributes


,attribute,level,topic,index,call_index,n_turns,total_words
0,certainty_seeking,high,wedding invitation for an estranged parent,0,1,10,382
1,certainty_seeking,low,mourning ritual after a cremation,0,1,8,366
2,certainty_seeking,neutral,toddler nap transition,0,1,8,313


## 3 · Coverage & naming consistency

Counts per attribute/level, and a check of the level-name vocabulary used by each attribute.

In [3]:
coverage = df.groupby(["attribute", "level"]).size().rename("n").reset_index()
coverage_pivot = coverage.pivot(index="attribute", columns="level", values="n")
coverage_pivot["total"] = coverage_pivot.sum(axis=1)
coverage_pivot

level,high,low,medium,neutral,total
attribute,,,,,
certainty_seeking,70.0,70.0,NaN,70.0,210.0
gullibility,72.0,72.0,72.0,NaN,216.0
rationality,72.0,72.0,72.0,NaN,216.0
seriousness,72.0,72.0,72.0,NaN,216.0


In [4]:
levels_by_attr = df.groupby("attribute")["level"].unique().apply(lambda a: sorted(a))
levels_by_attr

attribute
certainty_seeking    [high, low, neutral]
gullibility           [high, low, medium]
rationality           [high, low, medium]
seriousness           [high, low, medium]
Name: level, dtype: object

**Naming inconsistency**: `gullibility`, `rationality`, and `seriousness` all use
`{low, medium, high}` for their three levels, but `certainty_seeking` uses `{low, neutral,
high}` — `"neutral"` instead of `"medium"`. Any downstream loader that assumes a uniform
`{low, medium, high}` schema across attributes will silently mis-map or drop
`certainty_seeking`'s mid-level rows.

`certainty_seeking` is also slightly smaller (70/level, 210 total) than the other three
(72/level, 216 total) — one fewer generation batch, confirmed below.

## 4 · Batch structure sanity check

Conversations are generated in batches of 6 (`call_index`): one API call produces 2
conversations at each of the 3 levels, with **independently sampled topics per level**
(the generation prompt asks for equal topic variety across levels, not matched topics per
index). Confirm the batch size is a clean 6 everywhere, and show that topics are not
paired across levels for the same `index`.

In [5]:
call_sizes = df.groupby(["attribute", "call_index"]).size()
print("Calls with group size != 6:", (call_sizes != 6).sum(), "/", len(call_sizes))

sample = df[df.attribute == "gullibility"].sort_values(["call_index", "level"])
sample[sample.call_index.isin([1, 2])][["call_index", "index", "level", "topic"]]

Calls with group size != 6: 0 / 143


,call_index,index,level,topic
210,1,0,high,intermittent car starting
243,1,1,high,history of ritual bells
211,1,0,low,island ferry connection
244,1,1,low,donepezil and vivid dreams
212,1,0,medium,craft-market tax records
245,1,1,medium,career-change interview
276,2,2,high,Stretching a grocery budget
309,2,3,high,Sudden flashes and floaters
277,2,2,low,Backcountry water treatment
310,2,3,low,Salary negotiation research


In [6]:
matched = (
    df.groupby(["attribute", "index"])["topic"]
    .nunique()
    .eq(1)
)
print("Indices where topic is identical across all levels of an attribute:")
matched.groupby("attribute").sum().astype(int)

Indices where topic is identical across all levels of an attribute:


attribute
certainty_seeking    0
gullibility          0
rationality          0
seriousness          0
Name: topic, dtype: int64

No index has a matching topic across its three levels, for any attribute — topics vary
freely and independently per level rather than being told at three intensities. Good for
avoiding a shared-topic shortcut, but it also means this dataset can't support a
matched-topic contrastive analysis as-is.

## 5 · Structural integrity

Role alternation, exact-duplicate transcripts, and within-level topic collisions.

In [7]:
print("Missing .txt files:", (~df.txt_exists).sum())
print("Conversations violating strict HUMAN/ASSISTANT alternation:", (~df.alt_ok).sum())

df["text_hash"] = df["text"].apply(lambda t: hashlib.md5(t.encode()).hexdigest())
dup_hashes = df["text_hash"].value_counts()
print("Exact-duplicate transcripts:", (dup_hashes > 1).sum())

dup_topics = (
    df.groupby(["attribute", "level", df["topic"].str.lower()])
    .size()
    .loc[lambda s: s > 1]
)
print("\nWithin-level duplicate topics by attribute:")
dup_topics.groupby(level=0).size()

Missing .txt files: 0
Conversations violating strict HUMAN/ASSISTANT alternation: 0
Exact-duplicate transcripts: 0

Within-level duplicate topics by attribute:


attribute
certainty_seeking    2
gullibility          2
rationality          1
seriousness          4
dtype: int64

No missing files, no alternation violations, no exact-duplicate transcripts. Only 9
within-level duplicate topics total out of 858 conversations — negligible.

## 6 · Trait-word leakage

The generator's own `leak_stems_found` field already screens each conversation for its
trait's obvious stem words. Cross-check that against an independent keyword sweep over the
**user (HUMAN) turns only**, split by level, to see whether any leaked words are
concentrated in a way that would make the label trivially recoverable from surface
lexicon.

In [8]:
leaked = df[df["leak_stems_found"].apply(len) > 0]
print(f"Generator-flagged leaks: {len(leaked)}/{len(df)}")
leaked.groupby("attribute")["leak_stems_found"].apply(lambda s: Counter(w for t in s for w in t))

Generator-flagged leaks: 16/858


attribute                   
certainty_seeking  ambigu       1.0
                   uncertain    1.0
                   trust        NaN
                   reasonab     NaN
                   formal       NaN
                   silly        NaN
gullibility        ambigu       NaN
                   uncertain    NaN
                   trust        2.0
                   reasonab     NaN
                   formal       NaN
                   silly        NaN
rationality        ambigu       NaN
                   uncertain    NaN
                   trust        NaN
                   reasonab     8.0
                   formal       NaN
                   silly        NaN
seriousness        ambigu       NaN
                   uncertain    NaN
                   trust        NaN
                   reasonab     NaN
                   formal       3.0
                   silly        1.0
Name: leak_stems_found, dtype: float64

In [9]:
TRAIT_WORDS = {
    "gullibility": ["gullib", "trust", "skeptic", "sceptic", "credul", "naive", "naïve"],
    "rationality": ["rational", "reasonab", "logic", "irrational"],
    "seriousness": ["serious", "joke", "humor", "humour", "playful", "flippant"],
    "certainty_seeking": ["certain", "uncertain", "doubt", "confiden", "ambigu"],
}

HUMAN_RE = re.compile(r"human: (.*?)(?=\nassistant:|\nhuman:|\Z)", re.S)


def human_text(text: str) -> str:
    return " ".join(HUMAN_RE.findall(text.lower()))


rows = []
for attr, words in TRAIT_WORDS.items():
    sub = df[df.attribute == attr]
    for level, grp in sub.groupby("level"):
        htexts = grp["text"].apply(human_text)
        for w in words:
            n = htexts.str.contains(w).sum()
            if n:
                rows.append({"attribute": attr, "level": level, "word": w, "n_conversations": n})

leak_df = pd.DataFrame(rows)
leak_df.pivot_table(index=["attribute", "word"], columns="level", values="n_conversations", fill_value=0).astype(int)

level                        high  low  medium  neutral
attribute         word                                 
certainty_seeking ambigu        0    1       0        0
                  certain       0    1       0        0
                  confiden      0    0       0        1
                  uncertain     0    1       0        0
gullibility       trust         0    1       1        0
rationality       logic         1    0       1        0
                  rational      1    0       0        0
                  reasonab      3    1       4        0

Leakage is rare (at most a handful of conversations per level) and **not concentrated
in one level** for any attribute — `seriousness` has zero leaked trait words at all. This
is not a level-predictive lexical shortcut.

## 7 · Length confound across levels

The main quality concern: conversation length (word count) by level, per attribute.

In [10]:
len_stats = (
    df.groupby(["attribute", "level"])["total_words"]
    .agg(["count", "mean", "median", "std", "min", "max"])
    .round(1)
)
len_stats

count   mean  median    std  min  max
attribute         level                                         
certainty_seeking high        70  304.7   295.5   61.8  185  483
                  low         70  292.2   280.5   56.4  181  458
                  neutral     70  292.5   287.0   59.2  172  489
gullibility       high        72  352.5   338.0   74.4  196  587
                  low         72  411.5   406.5   77.7  267  728
                  medium      72  350.5   337.5   74.0  188  597
rationality       high        72  515.8   508.0   75.5  382  736
                  low         72  361.1   350.5   68.0  245  523
                  medium      72  409.7   413.5   66.9  261  573
seriousness       high        72  497.0   473.0  128.6  266  819
                  low         72  286.4   280.0   60.5  196  562
                  medium      72  350.4   341.0   76.8  240  603

In [11]:
level_order = {"low": 0, "medium": 1, "neutral": 1, "high": 2}
plot_df = df.copy()
plot_df["level_rank"] = plot_df["level"].map(level_order)

fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=False)
for ax, attr in zip(axes, ATTRS):
    sub = plot_df[plot_df.attribute == attr].sort_values("level_rank")
    order = sub.drop_duplicates("level").sort_values("level_rank")["level"].tolist()
    means = sub.groupby("level")["total_words"].mean().loc[order]
    colors = [PALETTE.get(l, "#8a8a8a") for l in order]
    ax.bar(order, means.values, color=colors)
    ax.set_title(attr)
    ax.set_ylabel("mean total words" if attr == ATTRS[0] else "")
fig.suptitle("Mean conversation length (words) by level, per attribute")
fig.tight_layout()
plt.show()

Length is **not balanced across levels**, and the direction is inconsistent across
attributes:

- `rationality` and `seriousness` both get markedly longer/more verbose as the level moves
  toward **high** (e.g. seriousness/high median far above seriousness/low, with roughly
  double the spread too).
- `gullibility` runs the **opposite** way — the skeptical, evidence-demanding "low" users
  produce the longest conversations, and "high" is shorter.
- `certainty_seeking` is comparatively well balanced across its three levels.

If these transcripts feed a linear probe or steering-vector pipeline, a classifier can
trivially pick up on turn count / sequence length rather than the intended behavioural
trait — especially for `rationality` and `seriousness`. Worth controlling for length when
training or evaluating probes on this data, or explicitly checking that probe accuracy
isn't just a proxy for length.

## 8 · Word-count distribution shape

Percentiles confirm the length differences above aren't driven by a handful of outliers —
whole distributions shift.

In [12]:
def pct_stats(s):
    return pd.Series({
        "min": s.min(), "p25": s.quantile(.25), "median": s.median(),
        "p75": s.quantile(.75), "max": s.max(), "std": s.std(),
    })

df.groupby(["attribute", "level"])["total_words"].apply(pct_stats).unstack().round(1)

min    p25  median    p75    max    std
attribute         level                                             
certainty_seeking high     185.0  261.2   295.5  345.2  483.0   61.8
                  low      181.0  251.8   280.5  326.8  458.0   56.4
                  neutral  172.0  258.2   287.0  320.5  489.0   59.2
gullibility       high     196.0  305.8   338.0  393.5  587.0   74.4
                  low      267.0  364.8   406.5  457.5  728.0   77.7
                  medium   188.0  296.8   337.5  403.2  597.0   74.0
rationality       high     382.0  462.0   508.0  546.2  736.0   75.5
                  low      245.0  306.0   350.5  411.5  523.0   68.0
                  medium   261.0  353.0   413.5  458.8  573.0   66.9
seriousness       high     266.0  405.8   473.0  583.0  819.0  128.6
                  low      196.0  247.5   280.0  314.0  562.0   60.5
                  medium   240.0  300.0   341.0  370.8  603.0   76.8

## 9 · Summary

- **Coverage**: 858 conversations total — 216 each for gullibility / rationality /
  seriousness, 210 for certainty_seeking (one fewer generation batch).
- **Naming**: `certainty_seeking` uses `low/neutral/high` instead of the `low/medium/high`
  used by the other three attributes — normalize before assuming a uniform schema.
- **Batch structure**: clean batches of 6 conversations (2 per level × 3 levels); topics
  are independently sampled per level, never matched across levels for the same index.
- **Integrity**: no missing files, no role-alternation violations, no exact-duplicate
  transcripts, only 9 within-level duplicate topics out of 858.
- **Trait-word leakage**: rare and spread across levels, not a usable lexical shortcut.
- **Length confound**: real and attribute-dependent — `rationality`/`seriousness` get
  longer toward "high", `gullibility` gets longer toward "low", `certainty_seeking` is
  roughly flat. This is the main caveat for any probing/steering analysis built on this
  corpus.